In [1]:
# Cell 1: Imports
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

from bs4 import BeautifulSoup
import time
import json
from datetime import datetime, timedelta
import re
import pandas as pd
import numpy as np
import random
from urllib.parse import urljoin

import undetected_chromedriver as uc

# For PostgreSQL
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

In [2]:
# Cell 2: Constants and Configuration

# --- URL và Scraping Params ---
TARGET_URL_BASE = "https://www.topcv.vn"
INITIAL_TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2" # Fresher HCM

MAX_PAGES_TO_SCRAPE = 1 # Chỉ cào 1 trang danh sách để test
# Đặt một số lớn để cố gắng cào hết các job trên trang đầu tiên, 
# hoặc bạn có thể bỏ qua biến này nếu vòng lặp for của bạn tự động duyệt hết các item tìm được
MAX_DETAILS_PER_PAGE = 18 # Đảm bảo nó lớn hơn số job thường có trên 1 trang (thường là 50)

# --- File Output ---
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
JSON_OUTPUT_FILENAME = f"topcv_detailed_jobs_data_{TIMESTAMP_STR}.json"

# --- HCM Districts ---
HCM_DISTRICTS_FULL_LIST = [
    "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8",
    "Quận 9", "Quận 10", "Quận 11", "Quận 12",
    "Thủ Đức", "Bình Thạnh", "Tân Bình", "Tân Phú", "Phú Nhuận", "Gò Vấp", "Bình Tân",
    "Hóc Môn", "Củ Chi", "Nhà Bè", "Bình Chánh", "Cần Giờ", "Thành phố Thủ Đức"
]
HCM_DISTRICTS_NORMALIZED_FOR_MATCH = [
    d.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
    for d in HCM_DISTRICTS_FULL_LIST
]

# --- Database Configuration ---
DB_TABLE_NAME = 'topcv_jobs_detailed'

print(f"Cell 2: Các hằng số và cấu hình đã được thiết lập.")
print(f"  URL ban đầu: {INITIAL_TARGET_URL}")
print(f"  Số trang tối đa cào: {MAX_PAGES_TO_SCRAPE}")
if 'MAX_DETAILS_PER_PAGE' in locals():
     print(f"  Số job chi tiết tối đa mỗi trang (nếu áp dụng): {MAX_DETAILS_PER_PAGE}")
print(f"  File JSON output: {JSON_OUTPUT_FILENAME}")
print(f"  Tên bảng DB: {DB_TABLE_NAME}")

Cell 2: Các hằng số và cấu hình đã được thiết lập.
  URL ban đầu: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
  Số trang tối đa cào: 1
  Số job chi tiết tối đa mỗi trang (nếu áp dụng): 18
  File JSON output: topcv_detailed_jobs_data_20250530_214022.json
  Tên bảng DB: topcv_jobs_detailed


In [3]:
# Cell 3: Selenium WebDriver Configuration and Initialization

print("Cell 3: Đang cấu hình và khởi tạo Selenium WebDriver...")
chrome_options_uc = Options()

# !!! QUAN TRỌNG: CẬP NHẬT USER-AGENT CỦA BẠN VÀO ĐÂY !!!
# Lấy từ chrome://version/ trong trình duyệt của bạn
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36" # THAY BẰNG USER AGENT CỦA BẠN
chrome_options_uc.add_argument(f"--user-agent={USER_AGENT}")
chrome_options_uc.add_argument("--window-size=1920,1080")
# chrome_options_uc.add_argument("--headless") # Bật khi đã debug xong và muốn chạy ẩn
# chrome_options_uc.add_argument("--disable-gpu")
# chrome_options_uc.add_argument("--no-sandbox") # Nếu chạy trên Linux/Docker
# chrome_options_uc.add_argument("--disable-dev-shm-usage") # Nếu chạy trên Linux/Docker
# chrome_options_uc.add_argument('--disable-blink-features=AutomationControlled') # Cố gắng che giấu thêm

driver = None # Khởi tạo driver là None
try:
    print("  Đang khởi tạo WebDriver bằng undetected_chromedriver...")
    driver = uc.Chrome(options=chrome_options_uc, headless=False) # headless=True để chạy ẩn khi ổn định
    print("  WebDriver đã được khởi tạo thành công!")
except Exception as e:
    print(f"  Lỗi khi khởi tạo WebDriver: {e}")
    print("  Hãy đảm bảo undetected_chromedriver đã được cài đặt và phiên bản Chrome/Chromium tương thích.")
    driver = None # Đặt lại là None nếu lỗi

Cell 3: Đang cấu hình và khởi tạo Selenium WebDriver...
  Đang khởi tạo WebDriver bằng undetected_chromedriver...
  WebDriver đã được khởi tạo thành công!


In [4]:
# Cell 4: Helper Function - Parse Location Details (from List Page item)

def parse_location_details(location_label_tag, hcm_districts_full, hcm_districts_normalized):
    """Trích xuất city_main và district từ thẻ label trên trang danh sách."""
    city_main, district, location_raw_from_span = None, None, None
    if not location_label_tag: return city_main, district, location_raw_from_span

    location_span_tag = location_label_tag.find('span', class_='city-text')
    if location_span_tag: location_raw_from_span = location_span_tag.get_text(strip=True)

    tooltip_attr = location_label_tag.get('data-original-title')
    if tooltip_attr:
        tooltip_soup = BeautifulSoup(tooltip_attr, 'html.parser')
        p_tag_in_tooltip = tooltip_soup.find('p')
        if p_tag_in_tooltip:
            locations_in_tooltip, current_segment = [], ""
            for content_part in p_tag_in_tooltip.contents:
                if isinstance(content_part, str): current_segment += content_part.strip()
                elif content_part.name == 'br':
                    if current_segment: locations_in_tooltip.append(current_segment)
                    current_segment = ""
            if current_segment: locations_in_tooltip.append(current_segment)
            
            districts_found_in_hcm, is_hcm_primary = [], False
            for loc_detail_str in locations_in_tooltip:
                loc_detail_str_lower = loc_detail_str.lower()
                if "hồ chí minh" in loc_detail_str_lower or "hcm" in loc_detail_str_lower:
                    is_hcm_primary = True
                    district_match = re.search(r'(?:hồ chí minh|hcm)\s*:\s*(.+)', loc_detail_str, re.IGNORECASE)
                    if district_match:
                        district_name_raw = district_match.group(1).strip()
                        norm_name = district_name_raw.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
                        try: districts_found_in_hcm.append(hcm_districts_full[hcm_districts_normalized.index(norm_name)])
                        except ValueError: districts_found_in_hcm.append(district_name_raw)
            if is_hcm_primary:
                city_main = "Hồ Chí Minh"
                if districts_found_in_hcm: district = ", ".join(sorted(list(set(districts_found_in_hcm))))
    
    if city_main == "Hồ Chí Minh" and not district and location_raw_from_span:
        norm_span_text = location_raw_from_span.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
        if norm_span_text not in ["hồ chí minh", "hcm"]:
            try: district = hcm_districts_full[hcm_districts_normalized.index(norm_span_text)]
            except ValueError: pass
    elif not city_main and location_raw_from_span and ("hồ chí minh" in location_raw_from_span.lower() or "hcm" in location_raw_from_span.lower()):
        city_main = "Hồ Chí Minh"
        
    return city_main, district, location_raw_from_span

print("Cell 4: Hàm `parse_location_details` đã được định nghĩa.")

Cell 4: Hàm `parse_location_details` đã được định nghĩa.


In [5]:
# Cell 5: Helper Functions - Parse Job Detail Page Content

# ... (Hàm get_job_detail_page_source, parse_company_info_from_detail, 
#      parse_general_job_info_from_detail, parse_skills_categories_from_detail,
#      parse_application_deadline_from_detail giữ nguyên như phiên bản trước) ...

def get_job_detail_page_source(driver_instance, detail_url, wait_time=30): # Giữ wait_time = 30 giây
    """Điều hướng đến trang chi tiết job và trả về page source."""
    try:
        driver_instance.get(detail_url)
        WebDriverWait(driver_instance, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.job-detail__information-detail--content"))
        )
        time.sleep(random.uniform(2, 4)) 
        return driver_instance.page_source
    except TimeoutException:
        print(f"    Timeout ({wait_time}s) khi chờ element trên trang chi tiết: {detail_url}.")
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S%f") 
            filename_base = f"debug_timeout_{detail_url.split('/')[-1].split('?')[0]}_{timestamp}"
            if driver_instance: # Kiểm tra trước khi dùng
                driver_instance.save_screenshot(f"{filename_base}.png")
                print(f"      Đã lưu screenshot: {filename_base}.png")
                with open(f"{filename_base}.html", "w", encoding="utf-8") as f_html_err:
                    f_html_err.write(driver_instance.page_source)
                print(f"      Đã lưu page source: {filename_base}.html")
        except Exception as e_save_debug:
            print(f"      Không thể lưu thông tin debug khi timeout: {e_save_debug}")
    except Exception as e: 
        print(f"    Lỗi khi truy cập hoặc xử lý trang chi tiết {detail_url}: {e}")
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S%f")
            filename_base = f"debug_error_{detail_url.split('/')[-1].split('?')[0]}_{timestamp}"
            if driver_instance: 
                 driver_instance.save_screenshot(f"{filename_base}.png")
                 print(f"      Đã lưu screenshot khi lỗi: {filename_base}.png")
                 with open(f"{filename_base}.html", "w", encoding="utf-8") as f_html_err:
                     f_html_err.write(driver_instance.page_source)
                 print(f"      Đã lưu page source khi lỗi: {filename_base}.html")
        except Exception as e_save_debug:
            print(f"      Không thể lưu thông tin debug khi lỗi: {e_save_debug}")
    return None

def parse_company_info_from_detail(soup_detail):
     company_info = {'company_name_detail': None, 'company_scale': None, 'company_field': None, 'company_full_address': None}
     container = soup_detail.find("div", class_="job-detail__company--information")
     if not container: return company_info
     if name_tag := container.select_one("div.company-name-label a.name"): company_info['company_name_detail'] = name_tag.get_text(strip=True)
     if scale_item := container.find("div", class_="company-scale"): 
         if val_tag := scale_item.find("div", class_="company-value"): company_info['company_scale'] = val_tag.get_text(strip=True)
     if field_item := container.find("div", class_="company-field"):
         if val_tag := field_item.find("div", class_="company-value"): company_info['company_field'] = val_tag.get_text(strip=True)
     if address_item := container.find("div", class_="company-address"):
         if val_div := address_item.find("div", class_="company-value"):
             company_info['company_full_address'] = val_div.get('data-original-title', val_div.get_text(strip=True)) # Ưu tiên data-original-title
     return company_info

def parse_general_job_info_from_detail(soup_detail):
     general_info = {'job_level': None, 'education_level': None, 'quantity_needed': None, 'employment_type': None, 'gender_requirement': None}
     box = soup_detail.find("div", class_="job-detail__body-right--box-general")
     if not box: return general_info
     for group in box.find_all("div", class_="box-general-group"):
         if title_tag := group.find("div", class_="box-general-group-info-title"):
             if value_tag := group.find("div", class_="box-general-group-info-value"):
                 title, value = title_tag.get_text(strip=True), value_tag.get_text(strip=True)
                 if "Cấp bậc" in title: general_info['job_level'] = value
                 elif "Học vấn" in title: general_info['education_level'] = value
                 elif "Số lượng tuyển" in title: general_info['quantity_needed'] = value 
                 elif "Hình thức làm việc" in title: general_info['employment_type'] = value
                 elif "Giới tính" in title: general_info['gender_requirement'] = value
     return general_info

def parse_skills_categories_from_detail(soup_detail):
     skills_cats = {'related_job_categories': [], 'required_skills_tags': [], 'preferred_skills_tags': []}
     container = soup_detail.find("div", class_="job-detail__body-right--box-category")
     if not container: return skills_cats # Trả về list rỗng nếu không tìm thấy container
     for box in container.find_all("div", class_="box-category"):
         if title_tag := box.find("div", class_="box-title"):
             if tags_container := box.find("div", class_="box-category-tags"):
                 title = title_tag.get_text(strip=True)
                 if "Danh mục Nghề" in title: 
                     skills_cats['related_job_categories'] = [a.get_text(strip=True) for a in tags_container.select("a.box-category-tag")] or []
                 elif "Kỹ năng cần có" in title: 
                     skills_cats['required_skills_tags'] = [s.get_text(strip=True) for s in tags_container.select("span.box-category-tag")] or []
                 elif "Kỹ năng nên có" in title: 
                     skills_cats['preferred_skills_tags'] = [s.get_text(strip=True) for s in tags_container.select("span.box-category-tag")] or []
     return skills_cats

def parse_job_content_from_detail(soup_detail): # CẬP NHẬT PHẦN working_time_text
    content = {'job_description_text': None, 'job_requirements_text': None, 'job_benefits_text': None, 'working_time_text': None}
    content_container = soup_detail.select_one("div.job-detail__information-detail--content div.job-description")
    if not content_container: return content
    
    for item in content_container.find_all("div", class_="job-description__item", recursive=False):
        h3_tag = item.find("h3")
        item_content_div = item.find("div", class_="job-description__item--content")
        
        if h3_tag and item_content_div:
            section_title = h3_tag.get_text(strip=True)
            text_parts = []
            
            if "Thời gian làm việc" in section_title:
                # Xử lý riêng cho "Thời gian làm việc" để lấy các list item
                list_items = item_content_div.find_all("div", class_="job-description__item--content-list")
                if list_items:
                    for li in list_items:
                        if li_text := li.get_text(strip=True):
                            text_parts.append(li_text)
                elif item_content_div.get_text(strip=True): # Fallback nếu không có cấu trúc list
                     text_parts.append(item_content_div.get_text(strip=True))
                section_text = "\n".join(text_parts) if text_parts else None
                content['working_time_text'] = section_text
            else:
                # Xử lý chung cho các mục khác
                for element in item_content_div.children:
                    if isinstance(element, str) and element.strip(): text_parts.append(element.strip())
                    elif element.name == 'p' and element.get_text(strip=True): text_parts.append(element.get_text(strip=True))
                    elif element.name in ['ul', 'ol']:
                        for li_tag in element.find_all('li', recursive=False): 
                            if li_text_content := li_tag.get_text(strip=True): text_parts.append(f"- {li_text_content}")
                section_text = "\n".join(text_parts) if text_parts else None

                if "Mô tả công việc" in section_title: content['job_description_text'] = section_text
                elif "Yêu cầu ứng viên" in section_title: content['job_requirements_text'] = section_text
                elif "Quyền lợi" in section_title: content['job_benefits_text'] = section_text
    return content

def parse_application_deadline_from_detail(soup_detail):
     if deadline_tag := soup_detail.find("div", class_="job-detail__information-detail--actions-label"):
         if match := re.search(r'(\d{2}/\d{2}/\d{4})', deadline_tag.get_text(strip=True)):
             return match.group(1)
     return None

print("Cell 5: Các hàm helper parse trang chi tiết đã được định nghĩa (cập nhật parse_job_content_from_detail).")

Cell 5: Các hàm helper parse trang chi tiết đã được định nghĩa (cập nhật parse_job_content_from_detail).


In [6]:
# Cell 6: Main Parsing Function (Combines List Page Item and Detail Page Info)

def parse_job_item_combined(item_element_on_list, scrape_date, driver_instance_for_detail):
    job_data = {
        'scrape_date': scrape_date, 'job_title': None, 'job_url': None, 
        'company_name': None, 'salary': None, # QUAN TRỌNG: Khởi tạo salary ở đây
        'location_raw': None, 'city_main': None, 'district': None,
        'experience': None, 'post_date': None,
    }
    detail_url_from_href = None

    h3_title_tag = item_element_on_list.select_one('h3.title, h3.title.highlight')
    if h3_title_tag:
        title_a_tag = h3_title_tag.find('a', recursive=False)
        if title_a_tag:
            href_value = title_a_tag.get('href')
            if href_value:
                detail_url_from_href = urljoin(TARGET_URL_BASE, href_value)
                job_data['job_url'] = detail_url_from_href
            job_title_span = title_a_tag.find('span')
            if job_title_span:
                original_title = job_title_span.get('data-original-title')
                job_data['job_title'] = original_title.strip() if original_title and original_title.strip() else job_title_span.get_text(strip=True)
            elif title_a_tag.get_text(strip=True):
                 job_data['job_title'] = title_a_tag.get_text(strip=True)

    if not detail_url_from_href: return None 

    if company_a := item_element_on_list.select_one('a.company, a.company.job-pro'):
        if company_name_span := company_a.find('span', class_='company-name'):
            job_data['company_name'] = company_name_span.get_text(strip=True)

    # Cố gắng lấy salary từ list page
    if salary_tag := item_element_on_list.find(['span','div','p','label'], class_='title-salary'): # Thêm các tag có thể
        job_data['salary'] = salary_tag.get_text(strip=True).replace('\\n','').strip()
    # Nếu không tìm thấy, job_data['salary'] sẽ giữ giá trị None đã khởi tạo

    if location_label := item_element_on_list.find('label', class_='address truncate'):
        city, dist, raw_loc = parse_location_details(location_label, HCM_DISTRICTS_FULL_LIST, HCM_DISTRICTS_NORMALIZED_FOR_MATCH)
        job_data['city_main'] = city
        job_data['district'] = dist
        job_data['location_raw'] = raw_loc
    
    if exp_label := item_element_on_list.find('label', class_='exp'):
        if exp_span := exp_label.find('span'): job_data['experience'] = exp_span.get_text(strip=True)

    if date_label := item_element_on_list.find('label', class_='address mobile-hidden label-update'):
        job_data['post_date'] = date_label.get_text(strip=True).replace('Đăng','').strip()

    detail_html = get_job_detail_page_source(driver_instance_for_detail, detail_url_from_href, wait_time=30) 
    
    if detail_html:
        soup_detail = BeautifulSoup(detail_html, 'html.parser')
        job_data.update(parse_company_info_from_detail(soup_detail))
        job_data.update(parse_general_job_info_from_detail(soup_detail))
        job_data.update(parse_skills_categories_from_detail(soup_detail))
        job_data.update(parse_job_content_from_detail(soup_detail)) # Gọi hàm đã cập nhật
        job_data['application_deadline_date'] = parse_application_deadline_from_detail(soup_detail)
    else:
        print(f"    Không lấy được HTML trang chi tiết cho {detail_url_from_href}, các trường chi tiết sẽ bị thiếu.")
    return job_data

print("Cell 6: Hàm `parse_job_item_combined` đã được định nghĩa (đảm bảo khởi tạo key và salary).")

Cell 6: Hàm `parse_job_item_combined` đã được định nghĩa (đảm bảo khởi tạo key và salary).


In [7]:
# Cell 7: Helper Function - Pagination Info Extraction

def extract_max_pages_from_list_soup(soup_list_page):
    """Trích xuất tổng số trang từ soup của trang danh sách."""
    try:
        if paginate_text_span := soup_list_page.select_one("nav.box-pagination span#job-listing-paginate-text"):
            if match := re.search(r'/\s*(\d+)\s*trang', paginate_text_span.get_text(strip=True)):
                return int(match.group(1))
    except Exception as e: print(f"  Lỗi khi trích xuất tổng số trang: {e}")
    return 1 # Mặc định là 1 nếu không tìm thấy

print("Cell 7: Hàm `extract_max_pages_from_list_soup` đã được định nghĩa.")

Cell 7: Hàm `extract_max_pages_from_list_soup` đã được định nghĩa.


In [8]:
# Cell 8: Main Scraping Loop

all_scraped_jobs_data = []
current_page_num = 1
actual_total_pages = 1 
current_list_page_url = INITIAL_TARGET_URL

if driver:
    print(f"Cell 8: Bắt đầu quá trình cào dữ liệu từ: {INITIAL_TARGET_URL}")
    while current_page_num <= MAX_PAGES_TO_SCRAPE : 
        print(f"\n--- Đang cào trang danh sách: {current_page_num} ---")
        
        list_page_html_source = None
        try:
            driver.get(current_list_page_url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "box-job-list")))
            list_page_html_source = driver.page_source
        except Exception as e:
            print(f"  Lỗi khi truy cập URL trang danh sách {current_list_page_url}: {e}")
            break 

        if list_page_html_source:
            soup_list = BeautifulSoup(list_page_html_source, 'html.parser')
            
            if current_page_num == 1:
                max_pages_on_site = extract_max_pages_from_list_soup(soup_list) # Gọi hàm từ Cell 7
                if max_pages_on_site > 1: actual_total_pages = max_pages_on_site
                print(f"  Tổng số trang trên site (ước tính): {actual_total_pages}")
            
            job_items_on_list = soup_list.select('div.box-job-list div.job-item-search-result')
            print(f"  Tìm thấy {len(job_items_on_list)} job items trên trang danh sách {current_page_num}.")

            current_scrape_timestamp = datetime.now().strftime('%Y-%m-%d')
            extracted_count_this_page = 0
            
            # Sẽ xử lý tất cả job items tìm được trên trang này (vì MAX_PAGES_TO_SCRAPE = 1)
            jobs_to_process_on_this_page = job_items_on_list 
            # Bỏ qua MAX_DETAILS_PER_PAGE nếu muốn test hết 1 trang
            # if 'MAX_DETAILS_PER_PAGE' in locals() and isinstance(MAX_DETAILS_PER_PAGE, int) and MAX_DETAILS_PER_PAGE > 0:
            #     jobs_to_process_on_this_page = job_items_on_list[:MAX_DETAILS_PER_PAGE]
            #     print(f"  Sẽ xử lý tối đa {len(jobs_to_process_on_this_page)} jobs chi tiết trên trang này.")


            for item_idx, item_el in enumerate(jobs_to_process_on_this_page):
                print(f"    Đang xử lý job item chi tiết thứ {item_idx + 1}/{len(jobs_to_process_on_this_page)}...")
                single_job_full_data = parse_job_item_combined(item_el, current_scrape_timestamp, driver) # Gọi hàm từ Cell 6
                if single_job_full_data:
                    all_scraped_jobs_data.append(single_job_full_data)
                    extracted_count_this_page +=1
                
                # TĂNG MẠNH THỜI GIAN NGHỈ GIỮA CÁC LẦN LẤY JOB CHI TIẾT
                sleep_detail_duration = random.uniform(15, 25) # Ví dụ: 15-25 giây
                if item_idx < len(jobs_to_process_on_this_page) - 1 : # Không sleep sau job cuối cùng của trang
                     print(f"      Nghỉ {sleep_detail_duration:.1f}s trước job chi tiết tiếp theo.")
                     time.sleep(sleep_detail_duration)
            
            print(f"  Đã xử lý và lấy chi tiết cho {extracted_count_this_page} jobs từ trang danh sách {current_page_num}.")

            # Logic dừng và chuyển trang
            if current_page_num >= actual_total_pages and actual_total_pages > 1:
                print(f"Đã cào đến trang cuối cùng trên site ({current_page_num}). Dừng.")
                break
            if current_page_num >= MAX_PAGES_TO_SCRAPE: # MAX_PAGES_TO_SCRAPE đang là 1
                 print(f"Đã cào đủ {MAX_PAGES_TO_SCRAPE} trang theo cấu hình. Dừng.")
                 break

            if next_page_a_tag := soup_list.select_one("ul.pagination a[rel='next'][data-href]"):
                current_list_page_url = urljoin(TARGET_URL_BASE, next_page_a_tag['data-href'])
            else:
                print("  Không tìm thấy link 'Next' page. Kết thúc cào.")
                break
        else:
            print(f"Không lấy được HTML trang danh sách {current_page_num}. Dừng.")
            break
            
        current_page_num += 1
        # Nếu MAX_PAGES_TO_SCRAPE > 1, thì thời gian nghỉ giữa các trang list cũng cần tăng
        if MAX_PAGES_TO_SCRAPE > 1 and current_page_num <= MAX_PAGES_TO_SCRAPE:
            sleep_list_page_duration = random.uniform(20, 35) # Ví dụ: 20-35 giây
            print(f"  Nghỉ {sleep_list_page_duration:.1f}s trước khi cào trang danh sách tiếp theo...")
            time.sleep(sleep_list_page_duration)
        
    print(f"\n--- HOÀN TẤT QUÁ TRÌNH SCRAPING ---")
    print(f"Tổng số jobs (đã lấy chi tiết) cào được: {len(all_scraped_jobs_data)}")
else:
    print("Cell 8: WebDriver không được khởi tạo. Bỏ qua scraping.")

Cell 8: Bắt đầu quá trình cào dữ liệu từ: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2

--- Đang cào trang danh sách: 1 ---
  Tổng số trang trên site (ước tính): 80
  Tìm thấy 50 job items trên trang danh sách 1.
    Đang xử lý job item chi tiết thứ 1/50...
      Nghỉ 24.9s trước job chi tiết tiếp theo.
    Đang xử lý job item chi tiết thứ 2/50...
      Nghỉ 19.3s trước job chi tiết tiếp theo.
    Đang xử lý job item chi tiết thứ 3/50...
      Nghỉ 18.0s trước job chi tiết tiếp theo.
    Đang xử lý job item chi tiết thứ 4/50...
    Timeout (30s) khi chờ element trên trang chi tiết: https://www.topcv.vn/brand/fecredit/tuyen-dung/nhan-vien-tu-van-tai-chinh-qua-dien-thoai-thu-nhap-trung-binh-tu-10-20-trieu-thang-j1263431.html?ta_source=JobSearchList_LinkDetail&u_sr_id=wTF6PohyuJ3j9p4z9WyDVhzR9oCBReTjUxOK9ny0_1748616025.
      Đã lưu screenshot: debug_timeout_nhan-vien-tu-van-tai-chinh-qua-dien-thoai-thu-nhap-trung-binh-tu-10-20-trie

KeyboardInterrupt: 

In [9]:
# Cell 9: Save Scraped Data to JSON and Close WebDriver

if all_scraped_jobs_data:
    try:
        print(f"\nCell 9: Đang lưu {len(all_scraped_jobs_data)} job items vào file JSON: {JSON_OUTPUT_FILENAME}")
        with open(JSON_OUTPUT_FILENAME, 'w', encoding='utf-8') as f_json:
            json.dump(all_scraped_jobs_data, f_json, ensure_ascii=False, indent=4)
        print(f"  Thành công! Dữ liệu đã được lưu vào file: '{JSON_OUTPUT_FILENAME}'")
        
        if len(all_scraped_jobs_data) > 0:
             print("\n  Ví dụ 1 job đầu tiên đã trích xuất:")
             first_job_sample = all_scraped_jobs_data[0]
             for key, value in first_job_sample.items():
                 if isinstance(value, list) and len(value) > 3: print(f"    {key}: {value[:3]}... ({len(value)-3} more)")
                 elif isinstance(value, str) and len(value) > 70: print(f"    {key}: {value[:70]}...")
                 else: print(f"    {key}: {value}")
    except Exception as e_save: print(f"  Lỗi khi lưu file JSON: {e_save}")
else: print("Cell 9: Không có dữ liệu job nào để lưu.")

if driver:
    try:
        print("Cell 9: Đang đóng WebDriver...")
        driver.quit()
        print("  WebDriver đã được đóng thành công.")
        driver = None 
    except Exception as e_quit: print(f"  Lỗi khi đóng WebDriver: {e_quit}")
else: print("Cell 9: WebDriver không tồn tại hoặc đã được đóng trước đó.")


Cell 9: Đang lưu 29 job items vào file JSON: topcv_detailed_jobs_data_20250530_214022.json
  Thành công! Dữ liệu đã được lưu vào file: 'topcv_detailed_jobs_data_20250530_214022.json'

  Ví dụ 1 job đầu tiên đã trích xuất:
    scrape_date: 2025-05-30
    job_title: Chuyên Viên Kinh Doanh Bất Động Sản
    job_url: https://www.topcv.vn/viec-lam/chuyen-vien-kinh-doanh-bat-dong-san/1741...
    company_name: CÔNG TY TNHH ĐẦU TƯ BĐS NGÔI NHÀ VIỆT
    salary: Thoả thuận
    location_raw: Hồ Chí Minh
    city_main: Hồ Chí Minh
    district: Quận 2
    experience: Không yêu cầu
    post_date: 6 ngày trước
    company_name_detail: CÔNG TY TNHH ĐẦU TƯ BĐS NGÔI NHÀ VIỆT
    company_scale: 100-499 nhân viên
    company_field: Bất động sản
    company_full_address: 69 Tạ Hiện, phường Thạnh Mỹ Lợi, TP. Thủ Đức, TP. HCM
    job_level: Nhân viên
    education_level: Trung cấp trở lên
    quantity_needed: 10 người
    employment_type: Toàn thời gian
    gender_requirement: None
    related_job_categorie

In [10]:
# Cell 10: Load Data from JSON into Pandas DataFrame and Initial Overview

df_jobs = pd.DataFrame() # Khởi tạo df rỗng
print(f"\nCell 10: Đang đọc dữ liệu từ file {JSON_OUTPUT_FILENAME}...")
if os.path.exists(JSON_OUTPUT_FILENAME):
    try:
        df_jobs = pd.read_json(JSON_OUTPUT_FILENAME, encoding='utf-8')
        if df_jobs.empty and all_scraped_jobs_data: 
             print("  !!! Cảnh báo: Đọc file JSON trả về DataFrame rỗng, nhưng biến `all_scraped_jobs_data` có dữ liệu. Tạo DataFrame từ biến này.")
             df_jobs = pd.DataFrame(all_scraped_jobs_data) 

        if not df_jobs.empty:
            print(f"  Đã đọc và tạo DataFrame với {len(df_jobs)} job items.")
            print("\n  Thông tin DataFrame (df_jobs.info()):")
            df_jobs.info() 
            print('\n  === 5 dòng dữ liệu đầu tiên (df_jobs.head()) ===')
            display(df_jobs.head())
            print("\n  --- Kiểm tra giá trị NULL cho mỗi cột ---")
            with pd.option_context('display.max_rows', None): print(df_jobs.isnull().sum().sort_values(ascending=False))
        else: print(f"  File {JSON_OUTPUT_FILENAME} được đọc nhưng DataFrame vẫn rỗng.")
    except Exception as e_read_json:
        print(f"  Lỗi khi đọc file JSON '{JSON_OUTPUT_FILENAME}': {e_read_json}")
        if all_scraped_jobs_data:
            df_jobs = pd.DataFrame(all_scraped_jobs_data)
            if not df_jobs.empty: print(f"  Đã tạo DataFrame từ `all_scraped_jobs_data` với {len(df_jobs)} dòng.")
else: print(f"  File {JSON_OUTPUT_FILENAME} không tồn tại. Không có dữ liệu để đọc.")


Cell 10: Đang đọc dữ liệu từ file topcv_detailed_jobs_data_20250530_214022.json...
  Đã đọc và tạo DataFrame với 29 job items.

  Thông tin DataFrame (df_jobs.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   scrape_date                29 non-null     object
 1   job_title                  29 non-null     object
 2   job_url                    29 non-null     object
 3   company_name               29 non-null     object
 4   salary                     29 non-null     object
 5   location_raw               29 non-null     object
 6   city_main                  29 non-null     object
 7   district                   28 non-null     object
 8   experience                 29 non-null     object
 9   post_date                  29 non-null     object
 10  company_name_detail        26 non-null     object
 11  company_

,scrape_date,job_title,job_url,company_name,salary,location_raw,city_main,district,experience,post_date,...,employment_type,gender_requirement,related_job_categories,required_skills_tags,preferred_skills_tags,job_description_text,job_requirements_text,job_benefits_text,working_time_text,application_deadline_date
0,2025-05-30,Chuyên Viên Kinh Doanh Bất Động Sản,https://www.topcv.vn/viec-lam/chuyen-vien-kinh...,CÔNG TY TNHH ĐẦU TƯ BĐS NGÔI NHÀ VIỆT,Thoả thuận,Hồ Chí Minh,Hồ Chí Minh,Quận 2,Không yêu cầu,6 ngày trước,...,Toàn thời gian,None,"[Kinh doanh/Bán hàng, Bất động sản/Xây dựng, S...",[],[],"- Chủ động tìm kiếm, xây dựng và phát triển ng...",- Tốt nghiệp trung cấp trở lên.\n- Không yêu c...,- Thu nhập hấp dẫn:Lương cứng lên đến 20 triệu...,Thứ 2 - Thứ 7 (từ 08:30 đến 17:30),23/06/2025
1,2025-05-30,Chăm Sóc Khách Hàng Tiktok Shop/App Du Lịch (T...,https://www.topcv.vn/viec-lam/cham-soc-khach-h...,CÔNG TY TNHH TƯ VẤN GIẢI PHÁP DOANH NGHIỆP TPC,10 - 13 triệu,Hồ Chí Minh,Hồ Chí Minh,Quận 12,Không yêu cầu,1 tuần trước,...,Toàn thời gian,None,[Chăm sóc khách hàng (Customer Service)/Vận hà...,"[Làm việc nhóm, Giao tiếp tốt, kỹ năng chăm só...",[],- Hỗ trợ khách hàng quốc tế sử dụng Ứng Dụng D...,"- Tiếng Anh tốt, tự tin làm việc với khách nướ...",Tổng thu nhập: 10.000.000 - 13.000.000 đ/tháng...,"Thời gian làm việc: 5 ngày/tuần, 9 tiếng/ca (b...",15/06/2025
2,2025-05-30,Cosmetics Sales Consultant,https://www.topcv.vn/viec-lam/cosmetics-sales-...,CÔNG TY TNHH MỘT THÀNH VIÊN NHÀ THUỐC TSURUHA ...,Thoả thuận,Hồ Chí Minh,Hồ Chí Minh,Quận 1,Không yêu cầu,1 ngày trước,...,Toàn thời gian,None,[Chăm sóc khách hàng (Customer Service)/Vận hà...,"[Tư Vấn Và Chăm Sóc Khách Hàng, Hiểu Biết Về L...",[],"•\tResearch, suggest needs and introduce cosme...","•\tPassionate, love makeup, skin care, have a ...","•\tCompetitive salary, attractive remuneration...",None,28/06/2025
3,2025-05-30,Nhân Viên Tư Vấn Tài Chính Qua Điện Thoại (Thu...,https://www.topcv.vn/brand/fecredit/tuyen-dung...,FE CREDIT,6.1 - 20 triệu,Hồ Chí Minh,Hồ Chí Minh,Tân Bình,Không yêu cầu,2 tuần trước,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-05-30,Nhân Viên Kinh Doanh/ Tư Vấn/ Sales Bất Động S...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,CÔNG TY CỔ PHẦN DỊCH VỤ BẤT ĐỘNG SẢN ĐẤT XANH,Tới 20 triệu,Hồ Chí Minh,Hồ Chí Minh,Thủ Đức,Không yêu cầu,2 tuần trước,...,Toàn thời gian,None,"[Kinh doanh/Bán hàng, Bất động sản/Xây dựng, S...",[],[],Chúng tôi không chỉ tìm kiếm một nhân sự – chú...,"- Từ 20 tuổi trở lên, tốt nghiệp THPT (ưu tiên...",THU NHẬP & CHÍNH SÁCH ĐÃI NGỘ\n- Lương cứng cố...,None,30/05/2025



  --- Kiểm tra giá trị NULL cho mỗi cột ---
gender_requirement           19
working_time_text            14
company_name_detail           3
company_field                 3
quantity_needed               3
education_level               3
job_level                     3
company_scale                 3
company_full_address          3
related_job_categories        3
employment_type               3
job_description_text          3
job_requirements_text         3
preferred_skills_tags         3
required_skills_tags          3
job_benefits_text             3
application_deadline_date     3
district                      1
post_date                     0
scrape_date                   0
experience                    0
location_raw                  0
city_main                     0
salary                        0
company_name                  0
job_title                     0
job_url                       0
dtype: int64


In [11]:
# Cell 11: Data Processing Helper Functions (Salary, Experience from List Page)

def parse_salary_string_for_list_data(salary_text):
    """Parse salary string (thường từ trang list) thành min, max, currency, type."""
    if pd.isna(salary_text): return np.nan, np.nan, 'triệu VNĐ', 'Không xác định'
    text_original, text_lower = str(salary_text), str(salary_text).lower()
    min_sal, max_sal, currency, sal_type = np.nan, np.nan, 'triệu VNĐ', 'Không xác định'

    if "thoả thuận" in text_lower:
        sal_type = "Thoả thuận"; return min_sal, max_sal, currency, sal_type

    if "usd" in text_lower: currency = "USD"; text_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("usd",""))
    elif "triệu" in text_lower: text_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("triệu","").replace("vnd",""))
    else: text_extract = re.sub(r'[^0-9.,\-]', '', text_lower)
    
    nums_str = re.findall(r'\d+[\.,]?\d*', text_extract)
    nums_float = []
    for ns in nums_str:
        cleaned_ns = ns.replace('.', '').replace(',', '.') if ',' in ns and '.' not in ns[:ns.find(',')] else ns.replace(',', '')
        try: nums_float.append(float(cleaned_ns))
        except ValueError: pass
    nums = sorted(nums_float)

    if not nums: return min_sal, max_sal, currency, sal_type

    if "tới" in text_lower or "upto" in text_lower or ("đến" in text_lower and len(nums) == 1 and "từ" not in text_lower):
        sal_type = "Tối đa"; max_sal = nums[-1] if nums else np.nan
    elif "từ" in text_lower and not any(k in text_lower for k in ["đến", "tới"]) and "-" not in text_original:
        sal_type = "Tối thiểu"; min_sal = nums[0] if nums else np.nan
    elif (("-" in text_original or "đến" in text_lower) and len(nums) >= 2) or len(nums) >= 2: # Sửa điều kiện này để bắt cả trường hợp chỉ có 2 số mà không có "đến" hay "-"
        sal_type = "Khoảng"
        if len(nums) >= 2:
            min_sal, max_sal = nums[0], nums[-1]
        elif len(nums) == 1: # Nếu sau khi lọc chỉ còn 1 số cho "khoảng"
             min_sal = max_sal = nums[0]
             sal_type = "Cố định" # Hoặc "Khoảng không rõ ràng"
        if min_sal == max_sal and pd.notna(min_sal) : sal_type = "Cố định" # Nếu min = max thì là cố định
    elif len(nums) == 1:
        sal_type = "Cố định"; min_sal = max_sal = nums[0]
    return min_sal, max_sal, currency, sal_type

def standardize_experience_for_list_data(exp_str):
    """Chuẩn hóa experience string (thường từ trang list)."""
    if pd.isna(exp_str): return 'Không xác định'
    exp_lower = str(exp_str).lower()
    if any(k in exp_lower for k in ["không yêu cầu", "chưa có kinh nghiệm", "no experience", "fresher", "mới tốt nghiệp"]): return "Không yêu cầu"
    if m := re.search(r'(?:dưới|<)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"< {m.group(1)} năm"
    if m := re.search(r'(?:từ\s*)?(\d+)\s*(?:-|đến)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)}-{m.group(2)} năm"
    if m := re.search(r'^(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)} năm"
    if m := re.search(r'(?:trên|>)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"> {m.group(1)} năm"
    return exp_str # Trả về gốc nếu không khớp
print("Cell 11: Các hàm helper xử lý dữ liệu (`parse_salary_string_for_list_data`, `standardize_experience_for_list_data`) đã được định nghĩa.")

Cell 11: Các hàm helper xử lý dữ liệu (`parse_salary_string_for_list_data`, `standardize_experience_for_list_data`) đã được định nghĩa.


In [12]:
# Cell 12: Data Processing - Apply Helper Functions (cho dữ liệu từ List Page) - HOÀN CHỈNH

if not df_jobs.empty:
    print("\nCell 12: Đang xử lý các cột gốc từ trang list (salary, post_date, experience)...")

    # --- KHỞI TẠO CÁC CỘT SẼ ĐƯỢC TẠO RA TỪ XỬ LÝ (ĐỂ ĐẢM BẢO LUÔN TỒN TẠI) ---
    # Cho Salary từ list
    df_jobs['salary_min_mil_list'] = np.nan
    df_jobs['salary_max_mil_list'] = np.nan
    df_jobs['salary_currency_list'] = 'triệu VNĐ' # Mặc định
    df_jobs['salary_type_list'] = 'Không xác định' # Mặc định
    # Cho Post Date từ list
    df_jobs['calculated_post_date_list'] = pd.NaT 
    # Cho Experience từ list
    df_jobs['experience_standardized_list'] = 'Không xác định' 

    # --- 1. Xử lý cột 'salary' (cột gốc tên là 'salary' trong df_jobs) ---
    if 'salary' in df_jobs.columns:
        salary_cols_parsed_names = ['salary_min_mil_list', 'salary_max_mil_list', 'salary_currency_list', 'salary_type_list']
        temp_salary_parsed_df = df_jobs['salary'].apply(
            lambda x: pd.Series(parse_salary_string_for_list_data(x), index=salary_cols_parsed_names)
        )
        for col_name in salary_cols_parsed_names:
            df_jobs[col_name] = temp_salary_parsed_df[col_name]
        print(f"  Đã xử lý cột 'salary' (gốc từ list). Các cột được cập nhật: {', '.join(salary_cols_parsed_names)}")
    else:
        print("  Cảnh báo Cell 12: Cột 'salary' (gốc từ list) không tồn tại. Các cột salary_..._list sẽ giữ giá trị NaN/mặc định.")

    # --- 2. Xử lý 'post_date' (cột gốc tên là 'post_date' trong df_jobs) ---
    if 'post_date' in df_jobs.columns and 'scrape_date' in df_jobs.columns:
        for index, row in df_jobs.iterrows():
            date_str, scrape_str = row['post_date'], row['scrape_date'] 
            calc_date = pd.NaT 
            if pd.isna(date_str) or pd.isna(scrape_str): continue
            try: scrape_dt = datetime.strptime(str(scrape_str), '%Y-%m-%d')
            except ValueError: continue
            date_lower = str(date_str).lower()
            if "hôm qua" in date_lower: calc_date = scrape_dt - timedelta(days=1)
            elif "hôm nay" in date_lower: calc_date = scrape_dt
            elif "ngày trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(days=int(m.group(1)))
            elif "tuần trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(weeks=int(m.group(1)))
            elif "tháng trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(days=int(int(m.group(1)) * 30.4375))
            else: 
                try: calc_date = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
                except: pass 
                if pd.isna(calc_date):
                    try: calc_date = pd.to_datetime(date_str, errors='coerce')
                    except: pass                    
            if pd.notna(calc_date): 
                df_jobs.loc[index, 'calculated_post_date_list'] = calc_date.normalize() if isinstance(calc_date, pd.Timestamp) else calc_date.replace(hour=0,minute=0,second=0,microsecond=0)
        print("  Đã xử lý 'post_date' (gốc từ list). Cột được cập nhật: 'calculated_post_date_list'")
    else:
        print("  Cảnh báo Cell 12: Thiếu cột 'post_date' hoặc 'scrape_date'. 'calculated_post_date_list' sẽ là NaT.")

    # --- 3. Xử lý 'experience' (cột gốc tên là 'experience' trong df_jobs) ---
    if 'experience' in df_jobs.columns:
        df_jobs['experience_standardized_list'] = df_jobs['experience'].apply(standardize_experience_for_list_data)
        print("  Đã xử lý 'experience' (gốc từ list). Cột được cập nhật: 'experience_standardized_list'")
    else:
        print("  Cảnh báo Cell 12: Cột 'experience' (gốc từ list) không tồn tại. 'experience_standardized_list' sẽ là 'Không xác định'.")
    
    # --- 4. ĐỔI TÊN CÁC CỘT GỐC TỪ LIST PAGE (nếu tồn tại) ĐỂ CÓ HẬU TỐ _raw_list ---
    cols_to_rename_to_raw = {
        'company_name': 'company_name_raw_list', 
        'salary': 'salary_raw_list',         
        'location_raw': 'location_raw_list', 
        'city_main': 'city_main_raw_list',       
        'district': 'district_raw_list',         
        'experience': 'experience_raw_list', 
        'post_date': 'post_date_raw_list'    
    }
    
    actual_renamed_cols = []
    for old_name, new_name in cols_to_rename_to_raw.items():
        if old_name in df_jobs.columns:
            df_jobs.rename(columns={old_name: new_name}, inplace=True)
            actual_renamed_cols.append(new_name)
        else:
            # Nếu cột gốc không tồn tại, đảm bảo cột _raw_list vẫn được tạo với giá trị None
            # để schema DB không bị lỗi khi thiếu cột
            if new_name not in df_jobs.columns: # Chỉ tạo nếu chưa có
                 df_jobs[new_name] = None
                 print(f"  Cảnh báo Cell 12: Cột gốc '{old_name}' không tìm thấy. Tạo cột '{new_name}' với giá trị None.")
            
    if actual_renamed_cols:
        print(f"  Đã đổi tên các cột gốc từ list page thành dạng _raw_list: {', '.join(actual_renamed_cols)}")
        
    print("  Hoàn tất xử lý và đổi tên cột cho dữ liệu từ trang list.")
else:
    print("Cell 12: DataFrame 'df_jobs' rỗng. Bỏ qua xử lý.")


Cell 12: Đang xử lý các cột gốc từ trang list (salary, post_date, experience)...
  Đã xử lý cột 'salary' (gốc từ list). Các cột được cập nhật: salary_min_mil_list, salary_max_mil_list, salary_currency_list, salary_type_list
  Đã xử lý 'post_date' (gốc từ list). Cột được cập nhật: 'calculated_post_date_list'
  Đã xử lý 'experience' (gốc từ list). Cột được cập nhật: 'experience_standardized_list'
  Đã đổi tên các cột gốc từ list page thành dạng _raw_list: company_name_raw_list, salary_raw_list, location_raw_list, city_main_raw_list, district_raw_list, experience_raw_list, post_date_raw_list
  Hoàn tất xử lý và đổi tên cột cho dữ liệu từ trang list.


In [13]:
# Cell 13: Data Processing - Apply Transformations (cho dữ liệu từ Detail Page)

if not df_jobs.empty:
    print("\nCell 13: Đang xử lý bổ sung cho các cột từ trang chi tiết...")

    # 1. Chuẩn hóa 'quantity_needed'
    if 'quantity_needed' in df_jobs.columns:
        df_jobs['quantity_needed_parsed'] = df_jobs['quantity_needed'].apply(
            lambda x: int(re.search(r'\d+', str(x)).group(0)) if pd.notna(x) and isinstance(x, str) and re.search(r'\d+', str(x)) else (int(x) if pd.notna(x) and isinstance(x, (int, float)) else None)
        )
        print("  Đã xử lý 'quantity_needed'. Cột mới: 'quantity_needed_parsed'")
    else:
        df_jobs['quantity_needed_parsed'] = None # Đảm bảo cột tồn tại
        print("  Cảnh báo: Cột 'quantity_needed' không tồn tại. 'quantity_needed_parsed' được tạo với giá trị None.")

    # 2. Chuẩn hóa 'application_deadline_date'
    if 'application_deadline_date' in df_jobs.columns:
        df_jobs['application_deadline_dt'] = pd.to_datetime(df_jobs['application_deadline_date'], format='%d/%m/%Y', errors='coerce')
        print("  Đã xử lý 'application_deadline_date'. Cột mới: 'application_deadline_dt' (datetime)")
    else:
        df_jobs['application_deadline_dt'] = pd.NaT # Đảm bảo cột tồn tại
        print("  Cảnh báo: Cột 'application_deadline_date' không tồn tại. 'application_deadline_dt' được tạo với giá trị NaT.")

    # --- Hàm join_list_elements (đã định nghĩa ở lần sửa trước, đảm bảo nó có ở đây hoặc cell trước) ---
    def join_list_elements(data_list):
        if isinstance(data_list, list):
            return ', '.join(str(item) for item in data_list) if data_list else None
        elif isinstance(data_list, np.ndarray) and data_list.size == 0: return None
        elif pd.isna(data_list): return None
        return str(data_list) if data_list else None

    # 3. Join list các skills/categories thành string
    list_cols_to_join_str = ['related_job_categories', 'required_skills_tags', 'preferred_skills_tags']
    for col_name in list_cols_to_join_str:
        if col_name in df_jobs.columns:
            df_jobs[f'{col_name}_str'] = df_jobs[col_name].apply(join_list_elements)
            print(f"  Đã chuyển cột list '{col_name}' thành string '{col_name}_str'")
        else:
            df_jobs[f'{col_name}_str'] = None # Đảm bảo cột tồn tại
            print(f"  Cảnh báo: Cột list '{col_name}' không tồn tại. '{col_name}_str' được tạo với giá trị None.")
    
    # --- 4. Trích xuất kỹ năng từ text JD/Requirements ---
    MASTER_SKILL_LIST = [
        "python", "sql", "java", "scala", "r", "c++", "c#", "javascript", "typescript", "php", "swift", "kotlin", "go", "ruby", "rust",
        "pandas", "numpy", "scipy", "scikit-learn", "sklearn", "tensorflow", "keras", "pytorch", "opencv", "nltk", "spacy", "gensim",
        "spark", "pyspark", "hadoop", "mapreduce", "hive", "pig", "kafka", "flink", "storm",
        "airflow", "luigi", "azkaban", "nifi",
        "docker", "kubernetes", "k8s", "openshift", "ansible", "terraform", "jenkins", "git", "cicd", "ci/cd",
        "aws", "azure", "gcp", "cloud", "s3", "ec2", "lambda", "rds", "dynamodb", "redshift", "glue", "emr", "sagemaker", 
        "azure data factory", "azure databricks", "azure synapse", "google bigquery", "google dataflow", "google dataproc",
        "excel", "vba", "power bi", "powerbi", "tableau", "qlik", "qlikview", "qliksense", "google data studio", "superset", "metabase", "looker",
        "ssis", "ssas", "ssrs", "informatica", "talend", "datastage",
        "etl", "elt", "data pipeline", "data modeling", "data modelling", "data warehouse", "dwh", "data lake", "data mart", "olap",
        "data analysis", "data analyst", "business intelligence", "bi", "business analyst", "ba",
        "data visualization", "data mining", "machine learning", "ml", "deep learning", "dl", "artificial intelligence", "ai", "nlp", "natural language processing",
        "statistics", "statistical modeling", "A/B testing", "hypothesis testing",
        "api", "restful api", "graphql", "microservices",
        "linux", "unix", "bash", "shell scripting",
        "nosql", "mongodb", "cassandra", "redis", "elasticsearch", "neo4j",
        "agile", "scrum", "kanban",
        "english", "tiếng anh", "giao tiếp", "communication", "presentation", "trình bày", 
        "problem solving", "critical thinking", "analytical skills", "phân tích", "tư duy logic", "teamwork", "làm việc nhóm"
    ] 
    
    df_jobs['skills_from_jd_text_list'] = None 
    df_jobs['skills_from_jd_text_str'] = None  

    if 'job_description_text' in df_jobs.columns or 'job_requirements_text' in df_jobs.columns:
        def extract_master_skills(row, skill_list_master):
            text_jd = str(row.get('job_description_text','')) if pd.notna(row.get('job_description_text')) else ''
            text_req = str(row.get('job_requirements_text','')) if pd.notna(row.get('job_requirements_text')) else ''
            
            # Không cần kết hợp tag skill ở đây nữa vì chúng ta đã có cột riêng cho chúng
            text_combined = (text_jd + " " + text_req).lower()
            
            if not text_combined.strip(): return None
            
            found_skills = []
            for skill_keyword in skill_list_master:
                # Regex để khớp từ/cụm từ hoàn chỉnh, không phân biệt hoa thường
                pattern = r'\b' + re.escape(skill_keyword.lower()) + r'\b'
                if re.search(pattern, text_combined):
                    found_skills.append(skill_keyword) 
            
            return list(set(found_skills)) if found_skills else None
        
        df_jobs['skills_from_jd_text_list'] = df_jobs.apply(extract_master_skills, args=(MASTER_SKILL_LIST,), axis=1)
        
        if 'skills_from_jd_text_list' in df_jobs.columns: # Kiểm tra lại trước khi apply
             df_jobs['skills_from_jd_text_str'] = df_jobs['skills_from_jd_text_list'].apply(join_list_elements)
        print("  Đã trích xuất skills từ text JD. Cột mới: 'skills_from_jd_text_list', 'skills_from_jd_text_str'")
    else:
        print("  Cảnh báo: Không có 'job_description_text' hoặc 'job_requirements_text' để trích xuất skill từ text. 'skills_from_jd_text_str' sẽ là None.")

    print("  Hoàn tất xử lý dữ liệu từ trang chi tiết.")
else:
    print("Cell 13: DataFrame 'df_jobs' rỗng. Bỏ qua xử lý.")


Cell 13: Đang xử lý bổ sung cho các cột từ trang chi tiết...
  Đã xử lý 'quantity_needed'. Cột mới: 'quantity_needed_parsed'
  Đã xử lý 'application_deadline_date'. Cột mới: 'application_deadline_dt' (datetime)
  Đã chuyển cột list 'related_job_categories' thành string 'related_job_categories_str'
  Đã chuyển cột list 'required_skills_tags' thành string 'required_skills_tags_str'
  Đã chuyển cột list 'preferred_skills_tags' thành string 'preferred_skills_tags_str'
  Đã trích xuất skills từ text JD. Cột mới: 'skills_from_jd_text_list', 'skills_from_jd_text_str'
  Hoàn tất xử lý dữ liệu từ trang chi tiết.


In [14]:
# Cell 14: Database Operations - Schema, Connection, Insertion

load_dotenv() 

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_TABLE_NAME = 'topcv_jobs_detailed' 

CREATE_TABLE_FINAL_SCHEMA_SQL = f"""
CREATE TABLE IF NOT EXISTS {DB_TABLE_NAME} (
    job_id SERIAL PRIMARY KEY,
    job_title TEXT,
    job_url TEXT UNIQUE,
    
    company_name_raw_list TEXT,
    salary_raw_list TEXT,
    salary_min_mil_list NUMERIC(10, 2),
    salary_max_mil_list NUMERIC(10, 2),
    salary_currency_list TEXT,
    salary_type_list TEXT,
    location_raw_list TEXT,
    city_main_raw_list TEXT,
    district_raw_list TEXT,
    experience_raw_list TEXT,
    experience_standardized_list TEXT,
    post_date_raw_list TEXT,
    calculated_post_date_list DATE,
    scrape_date DATE,

    company_name_detail TEXT, 
    company_scale TEXT,
    company_field TEXT,
    company_full_address TEXT,

    job_level TEXT,
    education_level TEXT,
    quantity_needed_parsed INT,
    employment_type TEXT,
    gender_requirement TEXT,

    related_job_categories_str TEXT,
    required_skills_tags_str TEXT,
    preferred_skills_tags_str TEXT,
    skills_from_jd_text_str TEXT,     -- THÊM CỘT NÀY

    job_description_text TEXT,
    job_requirements_text TEXT,
    job_benefits_text TEXT,
    working_time_text TEXT,
    
    application_deadline_dt DATE,
    
    inserted_at TIMESTAMP WITHOUT TIME ZONE DEFAULT CURRENT_TIMESTAMP 
);
"""

if not df_jobs.empty:
    print(f"\nCell 14: Đang chuẩn bị lưu dữ liệu vào DB bảng '{DB_TABLE_NAME}'...")
    if not all([DB_NAME, DB_USER, DB_PASSWORD]):
        print("  Lỗi: Thiếu thông tin kết nối database. Kiểm tra file .env.")
    else:
        conn, cur = None, None
        try:
            conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
            conn.autocommit = False 
            cur = conn.cursor()
            print("  Kết nối DB thành công!")
            
            cur.execute(CREATE_TABLE_FINAL_SCHEMA_SQL) # Chạy schema mới
            print(f"  Bảng '{DB_TABLE_NAME}' đã sẵn sàng.")

            df_for_db_insert = df_jobs.copy()
            
            db_target_columns = [
                'job_title', 'job_url', 
                'company_name_raw_list', 'salary_raw_list', 
                'salary_min_mil_list', 'salary_max_mil_list', 'salary_currency_list', 'salary_type_list',
                'location_raw_list', 'city_main_raw_list', 'district_raw_list',
                'experience_raw_list', 'experience_standardized_list',
                'post_date_raw_list', 'calculated_post_date_list', 'scrape_date',
                'company_name_detail', 'company_scale', 'company_field', 'company_full_address',
                'job_level', 'education_level', 'quantity_needed_parsed', 'employment_type', 'gender_requirement',
                'related_job_categories_str', 'required_skills_tags_str', 'preferred_skills_tags_str',
                'skills_from_jd_text_str', # THÊM CỘT NÀY VÀO LIST
                'job_description_text', 'job_requirements_text', 'job_benefits_text', 'working_time_text',
                'application_deadline_dt'
            ]
            
            missing_cols_in_df = [col for col in db_target_columns if col not in df_for_db_insert.columns]
            if missing_cols_in_df:
                print(f"  !!! LỖI: Các cột sau không tìm thấy trong DataFrame để chèn vào DB: {missing_cols_in_df}")
                print(f"      Các cột hiện có trong DataFrame: {sorted(list(df_for_db_insert.columns))}")
                # Tạo các cột thiếu với giá trị None để tránh lỗi insert, nhưng cần kiểm tra lại logic ở các cell trước
                for col_to_add in missing_cols_in_df:
                    print(f"      Tạo cột '{col_to_add}' với giá trị None do bị thiếu.")
                    df_for_db_insert[col_to_add] = None
                # raise KeyError(f"Missing columns for DB insert: {missing_cols_in_df}") # Hoặc raise lỗi để dừng lại

            date_cols_for_db = ['calculated_post_date_list', 'scrape_date', 'application_deadline_dt']
            for col_dt_db in date_cols_for_db:
                if col_dt_db in df_for_db_insert.columns:
                    df_for_db_insert[col_dt_db] = pd.to_datetime(df_for_db_insert[col_dt_db], errors='coerce').dt.strftime('%Y-%m-%d')
                    df_for_db_insert[col_dt_db] = df_for_db_insert[col_dt_db].replace({pd.NaT: None, 'NaT': None, 'nan':None})

            list_of_tuples_for_db = [
                tuple(row[col_name] if pd.notna(row[col_name]) else None for col_name in db_target_columns)
                for _, row in df_for_db_insert.iterrows()
            ]

            if not list_of_tuples_for_db:
                print("  Không có dữ liệu đã xử lý để chèn vào database.")
            else:
                print(f"  Đã chuẩn bị {len(list_of_tuples_for_db)} dòng dữ liệu để chèn/cập nhật.")
                insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES %s ON CONFLICT (job_url) DO NOTHING;").format(
                    sql.Identifier(DB_TABLE_NAME),
                    sql.SQL(', ').join(map(sql.Identifier, db_target_columns))
                )
                
                execute_values(cur, insert_stmt.as_string(conn), list_of_tuples_for_db, page_size=100)
                print(f"  Đã chèn/bỏ qua (nếu trùng job_url): {cur.rowcount} dòng.")
                conn.commit()
                print("  Dữ liệu đã được commit vào database.")

        except psycopg2.Error as e_db: print(f"  Lỗi PostgreSQL: {e_db}"); conn.rollback() if conn else None
        except KeyError as e_key: print(f"  Lỗi KeyError khi chuẩn bị chèn vào DB: {e_key}")
        except Exception as e_gen: print(f"  Lỗi không xác định khi thao tác DB: {e_gen}"); conn.rollback() if conn else None
        finally:
            if cur: cur.close()
            if conn: conn.close(); print("  Đã đóng kết nối database.")
else:
    print("Cell 14: DataFrame 'df_jobs' rỗng. Không có gì để thao tác với database.")

print("\n--- HOÀN TẤT TOÀN BỘ QUÁ TRÌNH NOTEBOOK ---")


Cell 14: Đang chuẩn bị lưu dữ liệu vào DB bảng 'topcv_jobs_detailed'...
  Kết nối DB thành công!
  Bảng 'topcv_jobs_detailed' đã sẵn sàng.
  Đã chuẩn bị 29 dòng dữ liệu để chèn/cập nhật.
  Đã chèn/bỏ qua (nếu trùng job_url): 29 dòng.
  Dữ liệu đã được commit vào database.
  Đã đóng kết nối database.

--- HOÀN TẤT TOÀN BỘ QUÁ TRÌNH NOTEBOOK ---
